In [7]:
## 0. Settings: study window, folders and data sources
import os
import json
import calendar
import urllib.request
import urllib.parse

# 12 months: July 2024 -> June 2025
MONTHS = ["2024-07", "2024-08", "2024-09", "2024-10", "2024-11", "2024-12",
          "2025-01", "2025-02", "2025-03", "2025-04", "2025-05", "2025-06"]

RAW = "../data/raw"

STATIONS_URL = "https://data.ny.gov/api/views/39hk-dx4f/rows.csv?accessType=DOWNLOAD"
WEATHER_URL = "https://www.ncei.noaa.gov/access/services/data/v1?dataset=local-climatological-data-v2&stations=USW00094728&startDate={start}&endDate={end}&format=csv"
RIDERSHIP_API = "https://data.ny.gov/resource/{dataset}"
PAGE_SIZE = 1000000

# The MTA publishes hourly ridership as one dataset per period: our window needs both
def ridership_dataset_for(month):
    if month < "2025-01":
        return "wujg-7c2s"   # MTA Subway Hourly Ridership: 2020-2024
    else:
        return "5wq4-mkjj"   # MTA Subway Hourly Ridership: Beginning 2025

In [14]:
ridership_dataset_for("2024-08")

'wujg-7c2s'

In [8]:
## 1. Download one URL into one file (skip if it exists, .part until complete)
def download_file(url, target):
    if os.path.exists(target):
        print("already downloaded, skipped:", target)
        return
    partial = target + ".part"
    urllib.request.urlretrieve(url, partial)
    os.replace(partial, target)
    print("downloaded:", target)

In [9]:
## 2. Stations table (1 file)
os.makedirs(RAW + "/stations", exist_ok=True)
download_file(STATIONS_URL, RAW + "/stations/stations.csv")

already downloaded, skipped: ../data/raw/stations/stations.csv


In [10]:
## 3. Weather: one file per month
os.makedirs(RAW + "/weather", exist_ok=True)
for month in MONTHS:
    year = int(month[:4])
    month_number = int(month[5:7])
    last_day = calendar.monthrange(year, month_number)[1]
    url = WEATHER_URL.format(start=f"{month}-01", end=f"{month}-{last_day:02d}")
    download_file(url, RAW + f"/weather/weather_{month}.csv")

already downloaded, skipped: ../data/raw/weather/weather_2024-07.csv
already downloaded, skipped: ../data/raw/weather/weather_2024-08.csv
already downloaded, skipped: ../data/raw/weather/weather_2024-09.csv
already downloaded, skipped: ../data/raw/weather/weather_2024-10.csv
already downloaded, skipped: ../data/raw/weather/weather_2024-11.csv
already downloaded, skipped: ../data/raw/weather/weather_2024-12.csv
already downloaded, skipped: ../data/raw/weather/weather_2025-01.csv
already downloaded, skipped: ../data/raw/weather/weather_2025-02.csv
already downloaded, skipped: ../data/raw/weather/weather_2025-03.csv
already downloaded, skipped: ../data/raw/weather/weather_2025-04.csv
already downloaded, skipped: ../data/raw/weather/weather_2025-05.csv
already downloaded, skipped: ../data/raw/weather/weather_2025-06.csv


In [11]:
## 4. Ridership: one month, page by page, with a row-count check
def download_ridership_month(month):
    target = RAW + f"/ridership/ridership_{month}.csv"
    if os.path.exists(target):
        print("already downloaded, skipped:", target)
        return

    # which dataset, and which time range (the whole month)
    dataset = ridership_dataset_for(month)
    year = int(month[:4])
    month_number = int(month[5:7])
    last_day = calendar.monthrange(year, month_number)[1]
    where = (f"transit_timestamp >= '{month}-01T00:00:00' AND "
             f"transit_timestamp <= '{month}-{last_day:02d}T23:59:59'")

    # 1. ask the API how many rows to expect
    count_query = urllib.parse.urlencode({"$select": "count(*) AS n", "$where": where})
    count_url = RIDERSHIP_API.format(dataset=dataset) + ".json?" + count_query
    expected = int(json.load(urllib.request.urlopen(count_url))[0]["n"])
    print(month, ": expecting", expected, "rows from dataset", dataset)

    # 2. download page after page into the .part file
    partial = target + ".part"
    with open(partial, "wb") as out:
        offset = 0
        while offset < expected:
            page_query = urllib.parse.urlencode({"$where": where, "$order": ":id",
                                                 "$limit": PAGE_SIZE, "$offset": offset})
            page_url = RIDERSHIP_API.format(dataset=dataset) + ".csv?" + page_query
            response = urllib.request.urlopen(page_url)
            header = response.readline()
            if offset == 0:
                out.write(header)
            out.write(response.read())
            offset = offset + PAGE_SIZE
            print("   ", month, ":", min(offset, expected), "/", expected)

    # 3. check: one header line + one line per row
    with open(partial, "rb") as f:
        rows = sum(1 for line in f) - 1
    if rows != expected:
        raise RuntimeError(f"{month}: got {rows} rows, expected {expected}")
    os.replace(partial, target)
    print("downloaded:", target, "-", rows, "rows")

In [12]:
## 5. Ridership: all 12 months
os.makedirs(RAW + "/ridership", exist_ok=True)
for month in MONTHS:
    download_ridership_month(month)

already downloaded, skipped: ../data/raw/ridership/ridership_2024-07.csv
already downloaded, skipped: ../data/raw/ridership/ridership_2024-08.csv
already downloaded, skipped: ../data/raw/ridership/ridership_2024-09.csv
already downloaded, skipped: ../data/raw/ridership/ridership_2024-10.csv
already downloaded, skipped: ../data/raw/ridership/ridership_2024-11.csv
already downloaded, skipped: ../data/raw/ridership/ridership_2024-12.csv
already downloaded, skipped: ../data/raw/ridership/ridership_2025-01.csv
already downloaded, skipped: ../data/raw/ridership/ridership_2025-02.csv
already downloaded, skipped: ../data/raw/ridership/ridership_2025-03.csv
already downloaded, skipped: ../data/raw/ridership/ridership_2025-04.csv
already downloaded, skipped: ../data/raw/ridership/ridership_2025-05.csv
already downloaded, skipped: ../data/raw/ridership/ridership_2025-06.csv
